# 00 — Data Loading

Load all 14 raw data files from `data/raw/`, perform initial structure checks, convert text files to CSV, and save everything as Parquet files.

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname('__file__'), '..', "scripts"))
# sys.path.insert(0, "C:/Disertation/UoB-GeneTraceAI-25-26/src/scripts")
# Navigate up to 'src' then down to 'scripts'
from pathlib import Path
print(Path.cwd())
print(sys.path)

import pandas as pd
import numpy as np
from src.scripts.data_utils import preview, BASE

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)
pd.set_option('display.max_colwidth', 40)

D:\MSc Data Science\Dissertation\AstraZeneca\UoB-GeneTraceAI-25-26
['..\\scripts', 'C:\\Users\\Thiruvel A P\\AppData\\Local\\Programs\\Python\\Python312\\python312.zip', 'C:\\Users\\Thiruvel A P\\AppData\\Local\\Programs\\Python\\Python312\\DLLs', 'C:\\Users\\Thiruvel A P\\AppData\\Local\\Programs\\Python\\Python312\\Lib', 'C:\\Users\\Thiruvel A P\\AppData\\Local\\Programs\\Python\\Python312', 'D:\\MSc Data Science\\Dissertation\\AstraZeneca\\UoB-GeneTraceAI-25-26\\GTAI', '', 'D:\\MSc Data Science\\Dissertation\\AstraZeneca\\UoB-GeneTraceAI-25-26\\GTAI\\Lib\\site-packages']


## Convert Text Files to CSV

In [2]:
# GEO expression
print(BASE)

geo_expr = pd.read_csv(f"{BASE}/gene expression/3_GEOexpression.txt", sep="\t", low_memory=False)
geo_expr.to_csv(f"{BASE}/gene expression/3_GEOexpression.csv", index=False)

# GEO info
geo_info = pd.read_csv(f"{BASE}/nomenclature/10_GEOInfo.txt", sep="\t", low_memory=False)
geo_info.to_csv(f"{BASE}/nomenclature/10_GEOInfo.csv", index=False)

print("Text files converted to CSV.")

## Load All 14 Raw Datasets

In [ ]:
# 2. DepMap expression matrix
depmap_expr = pd.read_csv(
    f"{BASE}/gene expression/2_DepMap_OmicsExpressionAllGenesTPMLogp1Profile.csv",
    index_col=0)
preview(depmap_expr, "2. DepMap Expression (TPM log+1)")

In [ ]:
# 3. GEO expression
try:
    geo_expr = pd.read_csv(f"{BASE}/gene expression/3_GEOexpression.txt", sep="\t")
    if geo_expr.shape[1] == 1:
        raise ValueError("not tab separated")
except Exception:
    geo_expr = pd.read_csv(f"{BASE}/gene expression/3_GEOexpression.csv")
preview(geo_expr, "3. GEO Expression")

In [ ]:
# 4. Proteomics
proteomics = pd.read_csv(f"{BASE}/gene expression/4_Harmonized_MS_CCLE_Gygi_subsetted.csv")
preview(proteomics, "4. Proteomics — CCLE Gygi MS")

In [ ]:
# 5. Fusion genes
fusions = pd.read_csv(f"{BASE}/gene properties/5_OmicsFusionFilteredSupplementary.csv")
preview(fusions, "5. Gene Fusions")

In [ ]:
# 6. Somatic mutations
mutations = pd.read_csv(f"{BASE}/gene properties/6_OmicsSomaticMutationsProfile.csv", low_memory=False)
preview(mutations, "6. Somatic Mutations")

In [ ]:
# 7. Cellosaurus
cellosaurus = pd.read_csv(f"{BASE}/nomenclature/7_cellosaurus.csv")
preview(cellosaurus, "7. Cellosaurus — Cell Line Dictionary")

In [ ]:
# 8. DepMap omics profiles
depmap_profiles = pd.read_csv(f"{BASE}/nomenclature/8_DepMap_OmicsProfiles.csv")
preview(depmap_profiles, "8. DepMap Omics Profiles")

In [ ]:
# 9. DepMap sample info
sample_info = pd.read_csv(f"{BASE}/nomenclature/9_DepMap_sample_info.csv")
preview(sample_info, "9. DepMap Sample Info")

In [ ]:
# 10. GEO series info
geo_info = pd.read_csv(f"{BASE}/nomenclature/10_GEOInfo.csv")
preview(geo_info, "10. GEO Series Info")

In [ ]:
# 11. HPA cell line descriptions
hpa_desc = pd.read_csv(f"{BASE}/nomenclature/11_hpa_rna_celline_description.tsv", sep="\t")
preview(hpa_desc, "11. HPA Cell Line Descriptions")

In [ ]:
# 12. CCLE metabolomics
metabolomics = pd.read_csv(f"{BASE}/non gene expression/12_CCLE_metabolomics_20190502.csv")
preview(metabolomics, "12. CCLE Metabolomics")

In [ ]:
# 13. CCLE miRNA
mirna = pd.read_csv(f"{BASE}/non gene expression/13_CCLE_miRNA_20181103.gct", sep="\t", skiprows=2)
preview(mirna, "13. CCLE miRNA (GCT)")

In [ ]:
# 14. Global pathway signatures
signatures = pd.read_csv(f"{BASE}/non gene expression/14_OmicsGlobalSignatures.csv")
preview(signatures, "14. Omics Global Signatures")

## Summary Table of All Datasets

In [ ]:
all_dfs = {
    1:  ("HPA RNA",          "hpa_rna",          hpa_rna),
    2:  ("DepMap Expr",       "depmap_expr",       depmap_expr),
    3:  ("GEO Expr",          "geo_expr",          geo_expr),
    4:  ("Proteomics",        "proteomics",        proteomics),
    5:  ("Fusions",           "fusions",           fusions),
    6:  ("Mutations",         "mutations",         mutations),
    7:  ("Cellosaurus",       "cellosaurus",       cellosaurus),
    8:  ("DepMap Profiles",   "depmap_profiles",   depmap_profiles),
    9:  ("Sample Info",       "sample_info",       sample_info),
    10: ("GEO Info",          "geo_info",          geo_info),
    11: ("HPA Desc",          "hpa_desc",          hpa_desc),
    12: ("Metabolomics",      "metabolomics",      metabolomics),
    13: ("miRNA",             "mirna",             mirna),
    14: ("Signatures",        "signatures",        signatures),
}

rows = []
for num, (name, obj, df) in all_dfs.items():
    missing_pct = df.isnull().sum().sum() / df.size * 100
    worst_col = df.isnull().sum().idxmax()
    worst_pct = df.isnull().sum().max() / len(df) * 100
    rows.append({
        "#": num, "File": name, "Object": obj,
        "Rows": f"{df.shape[0]:,}", "Cols": f"{df.shape[1]:,}",
        "Missing %": f"{missing_pct:.1f}%",
        "Worst col": worst_col, "Worst col %": f"{worst_pct:.1f}%",
    })

pd.DataFrame(rows)

## Save All Files as Parquet

In [ ]:
import os

OUT = os.path.join(BASE, "parquet/raw_data")
os.makedirs(OUT, exist_ok=True)

files = [
    (1, "gene expression",     "1_4_hpa_rna_celline.tsv",                             "\t",  0,    None),
    (2, "gene expression",     "2_DepMap_OmicsExpressionAllGenesTPMLogp1Profile.csv", ",",   0,    0),
    (3, "gene expression",     "3_GEOexpression.csv",                                 "\t",  0,    None),
    (4, "gene expression",     "4_Harmonized_MS_CCLE_Gygi_subsetted.csv",             ",",   0,    None),
    (5, "gene properties",     "5_OmicsFusionFilteredSupplementary.csv",              ",",   0,    None),
    (6, "gene properties",     "6_OmicsSomaticMutationsProfile.csv",                  ",",   0,    None),
    (7, "nomenclature",        "7_cellosaurus.csv",                                   ",",   0,    None),
    (8, "nomenclature",        "8_DepMap_OmicsProfiles.csv",                          ",",   0,    None),
    (9, "nomenclature",        "9_DepMap_sample_info.csv",                            ",",   0,    None),
    (10, "nomenclature",       "10_GEOInfo.txt",                                      "\t",  0,    None),
    (11, "nomenclature",       "11_hpa_rna_celline_description.tsv",                  "\t",  0,    None),
    (12, "non gene expression","12_CCLE_metabolomics_20190502.csv",                   ",",   0,    None),
    (13, "non gene expression","13_CCLE_miRNA_20181103.gct",                          "\t",  2,    None),
    (14, "non gene expression","14_OmicsGlobalSignatures.csv",                        ",",   0,    None),
]

results = []
for num, folder, fname, sep, skiprows, index_col in files:
    src = os.path.join(BASE, folder, fname)
    stem = fname.rsplit(".", 1)[0]
    out_path = os.path.join(OUT, f"{stem}.parquet")
    try:
        df = pd.read_csv(
            src, sep=sep,
            skiprows=skiprows if skiprows > 0 else None,
            index_col=index_col, low_memory=False)
        if num == 3 and df.shape[1] == 1:
            df = pd.read_csv(src, low_memory=False)
        df.to_parquet(out_path, index=(index_col is not None))
        size_mb = os.path.getsize(out_path) / 1_048_576
        results.append({"#": num, "File": fname, "Shape": f"{df.shape[0]:,} x {df.shape[1]:,}", "Parquet MB": f"{size_mb:.1f}"})
    except Exception as e:
        results.append({"#": num, "File": fname, "Shape": "—", "Parquet MB": f"ERROR: {e}"})
        print(f"[{num:02d}] ERROR: {e}")

pd.DataFrame(results)